# HTR CREMMA Medieval — Exp 3 (optimisée) : fine-tuning grayscale + LR decay

**But :** percer le plafond CER ~26% des runs 1–5.

**Deux leviers combinés :**
1. **Grayscale (mode L)** — `train_clean.arrow` (le seul Arrow vérifié mode L, SHA `1bec767c…`).
   > C'est le levier MAJEUR. Tout le reste n'a d'effet qu'une fois le mismatch L/1 levé.
2. **Fine-tuning doux + LR decay réactif** — LR 5e-5 + warmup + **reduceonplateau** + freeze-backbone + `-u NFD`.
   > `reduceonplateau` baisse le LR **quand le CER stagne** — adapté au plateau observé dans tes courbes.
   > (Alternative `cosine` documentée en commentaire dans la cellule 6b.)

## ⚠️ TEST DÉCISIF à surveiller au stage 0
Dans les logs de la cellule 6b, **le warning suivant NE DOIT PLUS apparaître** :
```
WARNING: Neural network has been trained on mode L images,
         training set contains mode 1 data.
```
- **S'il disparaît** → le grayscale est bien pris en compte, le CER devrait descendre sous 15%.
- **S'il apparaît encore** → l'Arrow est binarisé, INUTILE de continuer (même plafond que runs 1–5). Arrêter et recompiler.

**Prérequis :** Kaggle Secrets `AWS_ACCESS_KEY_ID` + `AWS_SECRET_ACCESS_KEY` · Accelerator **GPU T4 x2**.

**Ordre d'exécution :** 1 → 2 → 6a → 6b → 7

In [ ]:
# Cellule 1 — Connexion S3
from kaggle_secrets import UserSecretsClient
import boto3

secrets = UserSecretsClient()
s3 = boto3.client(
    's3',
    aws_access_key_id=secrets.get_secret('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=secrets.get_secret('AWS_SECRET_ACCESS_KEY'),
    region_name='eu-west-3'
)
print('Connexion S3 OK')

In [ ]:
# Cellule 2 — Installer Kraken
import subprocess, sys, torch

print(f'PyTorch existant : {torch.__version__}')
print(f'CUDA dispo : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print(f'GPU : {torch.cuda.get_device_name(0)} (sm_{cap[0]}{cap[1]})')
    print('bf16 supporte (Ampere+)' if cap[0] >= 8 else 'fp16 uniquement')

TORCH_VER = torch.__version__
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'kraken', 'iso639-lang', f'torch=={TORCH_VER}',
], check=True)

print('\n--- Test imports Kraken ---')
test = subprocess.run(
    [sys.executable, '-c',
     'from kraken.lib.xml import XMLPage; from kraken.train import KrakenTrainer; print("Imports OK")'],
    capture_output=True, text=True
)
print(test.stdout)
if test.returncode != 0:
    print('ERREUR IMPORT :'); print(test.stderr[-1500:])

for pkg in ['kraken', 'torch', 'lightning', 'torchmetrics']:
    v = subprocess.run([sys.executable, '-c', f'import importlib.metadata as m; print(m.version("{pkg}"))'],
                       capture_output=True, text=True)
    print(f'{pkg}: {v.stdout.strip() or v.stderr.strip()}')

In [ ]:
# Cellule 6a — Telecharger les Arrow GRAYSCALE filtres + modele de base depuis S3
import os

os.makedirs('/kaggle/working/splits', exist_ok=True)
os.makedirs('/kaggle/working/models', exist_ok=True)

for s3_key, local in [
    ('splits/train_clean.arrow',                '/kaggle/working/splits/train_clean.arrow'),
    ('splits/dev_clean.arrow',                  '/kaggle/working/splits/dev_clean.arrow'),
    ('base-model/cremma-generic-1.0.1.mlmodel', '/kaggle/working/cremma_generic.mlmodel'),
]:
    print(f'Telechargement {s3_key}...')
    s3.download_file('htr-cremma-medieval', s3_key, local)
    print(f'OK : {local} ({os.path.getsize(local)/1024/1024:.1f} MB)')

with open('/kaggle/working/splits/train_binary.txt', 'w') as f:
    f.write('/kaggle/working/splits/train_clean.arrow\n')
with open('/kaggle/working/splits/dev_binary.txt', 'w') as f:
    f.write('/kaggle/working/splits/dev_clean.arrow\n')
print('Manifests OK')

# --- Verification SHA-256 : confirme qu'on a bien le bon Arrow grayscale ---
import hashlib
ATTENDU = '1bec767c9a87caa322b20dc054da85e161ab3e630c498eb1a35ae51d19348026'
h = hashlib.sha256()
with open('/kaggle/working/splits/train_clean.arrow', 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
sha = h.hexdigest()
print(f'\ntrain_clean.arrow SHA-256 : {sha}')
print('  -> CORRESPOND au grayscale verifie' if sha == ATTENDU
      else '  -> ⚠ NE CORRESPOND PAS — verifier que c\'est bien le bon fichier !')

In [ ]:
# Cellule 6b — Fine-tuning OPTIMISE (grayscale + LR doux + warmup + reduceonplateau + freeze + NFD)
import subprocess, os, time, datetime, re

cmd = [
    'ketos',
    '--device', 'cuda:0',
    '--workers', '4',
    '--precision', '16-mixed',
    '--threads', '4',
    'train',
    '-f', 'binary',
    '-t', '/kaggle/working/splits/train_binary.txt',
    '-e', '/kaggle/working/splits/dev_binary.txt',
    '-i', '/kaggle/working/cremma_generic.mlmodel',
    '--resize', 'union',
    '-o', '/kaggle/working/models/htr_cremma_exp3opt',
    '-q', 'early',
    '-N', '50',
    '--lag', '15',                    # patience early-stopping (> sched-patience)
    '--min-epochs', '5',
    '-B', '8',
    # --- Fine-tuning doux + LR decay reactif a la stagnation ---
    '--optimizer', 'AdamW',
    '-r', '0.00005',                  # LR de depart : preserve les poids du modele CREMMA
    '--warmup', '500',                # rampe progressive (en samples) -> evite d'ecraser l'acquis
    '--schedule', 'reduceonplateau',  # baisse le LR QUAND le CER stagne (adapte a ton plateau)
    '--sched-patience', '5',          # attendre 5 evals sans amelioration avant de reduire le LR
    '--freeze-backbone', '2000',      # gele le backbone au debut (stabilise sur peu de donnees)
    '-u', 'NFD',                      # coherent avec la GT chocomufin (NFD)
    '--augment',
]
# NOTE schedule : 'reduceonplateau' reduit le LR seulement quand val_accuracy stagne
#   pendant --sched-patience evals. Si tu preferes une descente prederminee, remplacer par :
#       '--schedule', 'cosine', '--cos-min-lr', '0.000001',
#   et retirer --sched-patience.

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['TERM'] = 'dumb'

print('Commande :', ' '.join(cmd))
print(f'Debut    : {datetime.datetime.now().strftime("%H:%M:%S")}')
print('-' * 60)

t_global = time.time(); t_epoch = time.time()
epoch_num = 0; best_acc = 0.0; best_epoch = 0
mode1_warning_seen = False
last_lr = None

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=0, env=env)
buf = ''
while True:
    ch = proc.stdout.read(1)
    if not ch:
        break
    if ch == '\r':
        print('\r' + buf, end='', flush=True); buf = ''
    elif ch == '\n':
        line = buf; print(line)

        # --- TEST DECISIF : le warning mode 1 doit avoir disparu ---
        if re.search(r'mode\s*1\s*data|contains mode 1', line, re.IGNORECASE):
            mode1_warning_seen = True
            print('  WARNING MODE 1 DETECTE — l\'Arrow est binarise, meme plafond a prevoir !')
            print('       -> envisager d\'ARRETER : recompiler depuis preprocessed_grayscale/')

        m = re.search(r'stage\s+(\d+)', line, re.IGNORECASE)
        if m:
            epoch_num = int(m.group(1)); t_epoch = time.time()

        # Tracer les reductions de LR (signal que reduceonplateau s'est declenche)
        m_lr = re.search(r'(?:lr|learning[_\s]?rate)[:\s=]+([\d.eE+-]+)', line, re.IGNORECASE)
        if m_lr:
            try:
                lr = float(m_lr.group(1))
                if last_lr is not None and lr < last_lr:
                    print(f'  >> LR reduit : {last_lr:.2e} -> {lr:.2e} (reduceonplateau declenche)')
                last_lr = lr
            except ValueError:
                pass

        m_acc = re.search(r'val_accuracy[:\s]+([\d.]+)', line, re.IGNORECASE)
        if m_acc:
            acc = float(m_acc.group(1))
            if acc > best_acc:
                best_acc = acc; best_epoch = epoch_num
            lr_txt = f' | lr={last_lr:.2e}' if last_lr else ''
            print(f'  [Epoch {epoch_num:>2}] val_accuracy={acc*100:.2f}% | CER={((1-acc)*100):.2f}% | '
                  f'duree={time.time()-t_epoch:.0f}s | total={(time.time()-t_global)/60:.1f}min{lr_txt}')
            print(f'  Meilleur : epoch {best_epoch} -> {best_acc*100:.2f}% acc / {(1-best_acc)*100:.2f}% CER')

        m_pat = re.search(r'patience\s+(\d+)/(\d+)', line, re.IGNORECASE)
        if m_pat:
            cur, tot = int(m_pat.group(1)), int(m_pat.group(2))
            print(f'  Patience early-stop : {cur}/{tot} — arret dans {tot-cur} eval(s)')
        buf = ''
    else:
        buf += ch

if buf:
    print(buf)
proc.wait()

print('-' * 60)
print(f'Fin      : {datetime.datetime.now().strftime("%H:%M:%S")}')
print(f'Duree    : {(time.time()-t_global)/60:.1f} min')
print(f'Meilleur : epoch {best_epoch} -> {best_acc*100:.2f}% acc / {(1-best_acc)*100:.2f}% CER')
print(f'Code retour : {proc.returncode}')
print()
if mode1_warning_seen:
    print('BILAN : warning mode 1 vu -> le grayscale n\'a PAS ete pris en compte.')
    print('  Le plafond ~26% est attendu. Verifier la compilation de l\'Arrow.')
else:
    print('BILAN : aucun warning mode 1 — grayscale OK. Le CER reflete le vrai potentiel.')

In [ ]:
# Cellule 7 — Uploader le modele final sur S3
import glob, datetime
from pathlib import Path

models_dir = '/kaggle/working/models'
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')

mlmodels    = glob.glob(f'{models_dir}/**/*.mlmodel', recursive=True)
safetensors = glob.glob(f'{models_dir}/**/*.safetensors', recursive=True)
checkpoints = glob.glob(f'{models_dir}/**/*.ckpt', recursive=True)

print('Fichiers trouves :')
print(f'  .mlmodel     : {mlmodels}')
print(f'  .safetensors : {safetensors}')
print(f'  .ckpt        : {checkpoints}')

uploaded = []
for f in mlmodels:
    nom = f'exp3opt_finetune_{timestamp}.mlmodel'
    s3.upload_file(f, 'htr-cremma-medieval', f'output/{nom}')
    print(f'Upload : s3://htr-cremma-medieval/output/{nom}'); uploaded.append(nom)

if not mlmodels:
    for f in safetensors:
        nom = f'exp3opt_finetune_{timestamp}.safetensors'
        s3.upload_file(f, 'htr-cremma-medieval', f'output/{nom}')
        print(f'Upload : s3://htr-cremma-medieval/output/{nom}'); uploaded.append(nom)

if not mlmodels and not safetensors and checkpoints:
    best = sorted(checkpoints,
                  key=lambda p: float(Path(p).stem.split('-')[-1]) if '-' in Path(p).stem else 0,
                  reverse=True)[0]
    nom = f'exp3opt_finetune_{timestamp}.ckpt'
    s3.upload_file(best, 'htr-cremma-medieval', f'output/{nom}')
    print(f'Upload (checkpoint) : s3://htr-cremma-medieval/output/{nom}'); uploaded.append(nom)

if not uploaded:
    print('ERREUR : aucun modele trouve dans', models_dir)
    for f in Path(models_dir).rglob('*'):
        print(f'  {f}')
else:
    print(f'OK — {len(uploaded)} fichier(s) uploade(s) sur S3')